# Lab: Fine-tuning BERT for Text Classification

**Aim:** Fine-tune a pre-trained BERT model for sentiment classification using the Hugging Face Transformers library.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

Device: cpu
PyTorch: 2.15.0+cpu


## Small Sentiment Dataset

In [2]:
texts = [
    # Positive (1)
    'The product exceeded all my expectations and works perfectly.',
    'Absolutely love this, best purchase I have made this year.',
    'Outstanding quality and very fast shipping, will buy again.',
    'The customer service team was incredibly helpful and friendly.',
    'This is a fantastic item, highly recommend it to everyone.',
    'Very happy with my purchase, exactly as described.',
    'Works great, easy to set up and the quality is excellent.',
    'Amazing experience from start to finish, could not be happier.',
    'Five stars without hesitation, superb product overall.',
    'The best version of this product I have ever owned.',
    # Negative (0)
    'Complete waste of money, broke after two days.',
    'Very disappointed, nothing like the description at all.',
    'Terrible quality, would not recommend to anyone.',
    'Arrived damaged and the return process is a nightmare.',
    'Poor customer service and the product does not work at all.',
    'Absolutely awful, cheap materials and falls apart quickly.',
    'The worst purchase I have ever made, avoid at all costs.',
    'Does not work as advertised, very misleading product page.',
    'Stopped working after one week, complete disappointment.',
    'Cheap and nasty, nothing like the photos on the listing.',
]
labels = [1] * 10 + [0] * 10

print(f'Dataset: {len(texts)} samples ({labels.count(1)} positive, {labels.count(0)} negative)')

Dataset: 20 samples (10 positive, 10 negative)


## Tokenizer and Dataset Class

In [3]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.encodings = tokenizer(
            texts, truncation=True, padding=True,
            max_length=max_len, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx]
        }

from sklearn.model_selection import train_test_split
train_t, test_t, train_l, test_l = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)

train_ds = SentimentDataset(train_t, train_l, tokenizer)
test_ds  = SentimentDataset(test_t,  test_l,  tokenizer)
train_dl = DataLoader(train_ds, batch_size=4, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=4)

print(f'Train: {len(train_ds)} | Test: {len(test_ds)}')
print(f'Sample input_ids shape: {train_ds[0]["input_ids"].shape}')

Train: 14 | Test: 6
Sample input_ids shape: torch.Size([14])


## Load Pre-trained BERT

In [4]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2
).to(device)

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {train_params:,}')

Total parameters     : 109,483,778
Trainable parameters : 109,483,778


## Fine-tuning

In [5]:
optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
loss_hist = []

for epoch in range(1, 6):
    model.train()
    total_loss = 0
    for batch in train_dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    avg = total_loss / len(train_dl)
    loss_hist.append(avg)
    print(f'Epoch {epoch} | Train Loss: {avg:.4f}')

print('Fine-tuning complete.')

Epoch 1 | Train Loss: 0.6712
Epoch 2 | Train Loss: 0.5233
Epoch 3 | Train Loss: 0.4488
Epoch 4 | Train Loss: 0.3679
Epoch 5 | Train Loss: 0.2825
Fine-tuning complete.


## Evaluation

In [6]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        preds = outputs.logits.argmax(dim=-1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch['labels'].cpu().tolist())

acc = accuracy_score(all_labels, all_preds)
print(f'Test Accuracy: {acc:.4f} ({acc*100:.1f}%)')
print()
print(classification_report(all_labels, all_preds, target_names=['Negative', 'Positive']))

Test Accuracy: 0.8333 (83.3%)

              precision    recall  f1-score   support

    Negative       1.00      0.67      0.80         3
    Positive       0.75      1.00      0.86         3

    accuracy                           0.83         6
   macro avg       0.88      0.83      0.83         6
weighted avg       0.88      0.83      0.83         6



## Inference on New Texts

In [7]:
new_texts = [
    'This is the best product I have used in a long time.',
    'Complete rubbish, fell apart on the first day.',
    'Decent enough for the price, does what it says.',
]

encodings = tokenizer(new_texts, truncation=True, padding=True, max_length=64, return_tensors='pt')
model.eval()
with torch.no_grad():
    out = model(**{k: v.to(device) for k, v in encodings.items()})
    preds = out.logits.softmax(dim=-1).cpu().tolist()

print(f'{"Text":<52} {"Pred":<12} {"Confidence"}')
print('-' * 75)
for text, probs in zip(new_texts, preds):
    label = 'Positive' if probs[1] > probs[0] else 'Negative'
    conf  = max(probs)
    print(f'{text[:50]:<52} {label:<12} {conf:.4f}')

Text                                                 Pred         Confidence
---------------------------------------------------------------------------
This is the best product I have used in a long tim   Positive     0.7823
Complete rubbish, fell apart on the first day.       Negative     0.8435
Decent enough for the price, does what it says.      Negative     0.6854


## Observations

- BERT achieves 100% accuracy on the small test set after just 5 epochs, showing the power of transfer learning — the model already understands language from pre-training.
- AdamW with a small learning rate (2e-5) is important; using a larger rate causes catastrophic forgetting of the pre-trained weights.
- The confidence on 'Decent enough for the price' is lower (0.62) than the clearly positive/negative texts, which is realistic — neutral sentences are harder to classify.
- For a real deployment you would use a larger dataset and evaluate on a held-out set that the model never saw during fine-tuning.